# Make an eid index

# Loading row indices

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import dask.dataframe as dd
from pathlib import Path
import pyarrow as pa


In [2]:
import os
import sys

root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)

In [3]:
from data_metadata_input import parquet_basic_imports
from env.parameters import P

In [ ]:
P.output_parquet_path

In [5]:
P.core_all_name

'core_all'

In [ ]:
# List of fields
dd_all = dd.read_parquet(f'''{P.output_parquet_path}/{P.core_all_name}''')
dd_all.columns

In [7]:
# Make a pandas eid from raw data
pd_eid_raw = dd_all[['eid']].compute()


In [ ]:
pd_eid_raw.count()

In [ ]:
# Load withdrawals
P.withdrawal_path

In [ ]:
P.input_file_dict.get("withdrawals").get("path")

In [ ]:
# Withdrawals
w_absolute_path = os.path.expanduser(P.input_file_dict.get("withdrawals").get("path"))
print(w_absolute_path)
with open(w_absolute_path, 'r') as file:
    id_list= [int(line.strip()) for line in file]
df_withdraw = pd.DataFrame({'eid': id_list})
df_withdraw.head(5)

In [17]:
df_eid = pd_eid_raw[~pd_eid_raw['eid'].isin(df_withdraw['eid'])]


In [ ]:
print(f'''Number of removed records: {pd_eid_raw[pd_eid_raw['eid'].isin(df_withdraw['eid'])].shape[0]}''')

In [ ]:
print(f'''Number of records: {df_eid.shape[0]}''')


In [20]:

# Any duplicates?
if df_eid.duplicated(subset=["eid"]).any():
    print("Duplicated eid found")
else:
    print("No duplicated eid")

No duplicated eid


In [21]:
# save as csv and parquet
df_eid.to_csv(f'''{P.processed_data_path}/csv/eid_included.csv''', index=False)


In [22]:
# Make a Dask dataframe
dd_eid = dd.from_pandas(df_eid, npartitions=2)

In [23]:
from util.parquet_maker import dask_to_parquet

In [ ]:
P.output_parquet_path

In [26]:
# Make a Parquet dataframe
dask_to_parquet(dd_eid, output_path=P.output_parquet_path, output_parquet_name="eid_included")

/Users/mehrdadmizani/Documents/Biobank_2023/processed_data/parquet/eid_included does not exist. Creating the folder
To parquet ...
parquet files saved in /Users/mehrdadmizani/Documents/Biobank_2023/processed_data/parquet/eid_included


In [ ]:
# test
dd_in= dd.read_parquet(f'''{P.output_parquet_path}/eid_included''')


In [ ]:
dd_in.head()